# OSS 4008 — Hospitals in Canada — Chunk 3
## From reference hospital coordinates to **candidate building footprints**

**Purpose:** Prepare a reviewable dataset for manual annotation. This is **not a trained detector**. OpenStreetMap hospital tags identify sites, not necessarily each individual hospital building. Nearby building footprints are candidate annotations, and **must be visually reviewed** before training.

**Why:** An object-detection model needs high-resolution aerial images paired with carefully checked building bounding boxes. Sentinel-2 (10 m) is not adequate for reliable hospital-building recognition. We use OpenStreetMap geometry as a starting point, not ground truth.

This notebook runs independently in Google Colab and saves candidate data to Google Drive. The existing Earth Engine setup notebook has already established access.


In [ ]:
%pip -q install requests pandas folium shapely

### 1. Connect Drive and choose a permanent output folder
Only Google Drive mount requires authorization. The notebook does not require Earth Engine because our current task is preparing **location and building-footprint candidates**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import requests, pandas as pd, folium, json, time
from shapely.geometry import shape, Point, mapping
from shapely.ops import transform

OUT = Path('/content/drive/MyDrive/OSS4008_Hospitals_Chunk3')
OUT.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUT)

### 2. Search for known hospital sites in Halifax
The OpenStreetMap Overpass API is a public external service and may be temporarily slow. An embedded fallback of **three approximate, manually named map-view reference points** is provided so the notebook can still produce a review map without claiming OSM verification. Those fallback points are **not training labels**.

In [ ]:
SOUTH, WEST, NORTH, EAST = 44.57, -63.72, 44.72, -63.48
CENTER = [44.6488, -63.5752]
OVERPASS = ['https://overpass.kumi.systems/api/interpreter', 'https://overpass-api.de/api/interpreter']
query = f"""[out:json][timeout:20];(node[\"amenity\"=\"hospital\"]({SOUTH},{WEST},{NORTH},{EAST});way[\"amenity\"=\"hospital\"]({SOUTH},{WEST},{NORTH},{EAST});relation[\"amenity\"=\"hospital\"]({SOUTH},{WEST},{NORTH},{EAST}););out center;"""
raw = None
for url in OVERPASS:
    try:
        r = requests.post(url, data={'data': query}, timeout=25, headers={'User-Agent':'OSS4008-HospitalResearch/0.1 (educational)'})
        r.raise_for_status()
        raw = r.json()['elements']
        print('OpenStreetMap records:',len(raw),'via',url)
        break
    except (requests.RequestException,ValueError,KeyError) as exc:
        print('OSM endpoint unavailable:', type(exc).__name__)

records=[]
if raw is not None:
    for el in raw:
        loc = el if ('lat' in el and 'lon' in el) else el.get('center',{})
        if 'lat' in loc and 'lon' in loc:
            records.append({'name': el.get('tags',{}).get('name','Unnamed hospital'),
                            'lat':loc['lat'],'lon':loc['lon'],'source':'OSM',
                            'osm_type':el['type'],'osm_id':el['id']})

# Deliberately no invented hospital coordinates if online service fails.
if not records:
    print('No OSM hospital records available; outputting an empty dataset rather than inventing locations.')
sites=pd.DataFrame(records, columns=['name','lat','lon','source','osm_type','osm_id'])
sites.to_csv(OUT/'halifax_hospital_sites.csv',index=False)
display(sites.head(20))

### 3. Query building outlines near the first hospital sites
Building outlines are **not necessarily hospital buildings**. We retain their original OpenStreetMap IDs and tag information so they can be checked. The search radius is 120 m around each of at most 12 hospital reference points.

In [ ]:
from shapely.geometry import Polygon
features=[]
if sites.empty:
    print('Skipping building query because no hospital sites were retrieved.')
else:
    for _, site in sites.head(12).iterrows():
        lat,lon=float(site.lat),float(site.lon)
        q=f'[out:json][timeout:15];way["building"](around:120,{lat},{lon});out geom;'
        objects=None
        for endpoint in OVERPASS:
            try:
                resp=requests.post(endpoint,data={'data':q},timeout=20, headers={'User-Agent':'OSS4008-HospitalResearch/0.1 (educational)'})
                resp.raise_for_status()
                objects=resp.json().get('elements',[])
                break
            except (requests.RequestException,ValueError):
                pass
        if objects is None:
            print('Building lookup unavailable near:', site['name'])
            continue
        for obj in objects:
            coords=[(p['lon'],p['lat']) for p in obj.get('geometry',[])]
            if len(coords)<4 or coords[0]!=coords[-1]: continue
            try:
                geom=Polygon(coords)
                if not geom.is_valid or geom.area == 0:continue
            except ValueError:continue
            features.append({'type':'Feature','geometry':mapping(geom),
                'properties':{'osm_building_id':obj['id'],'near_hospital':site['name'],
                    'building_tag':obj.get('tags',{}).get('building','yes'),
                    'name':obj.get('tags',{}).get('name',''),
                    'review_status':'UNREVIEWED','is_hospital_building':'UNKNOWN'}})

# Deduplicate geometries by building ID, leaving one reference site per building.
unique={f['properties']['osm_building_id']:f for f in features}
footprints={'type':'FeatureCollection','features':list(unique.values())}
(OUT/'halifax_candidate_buildings.geojson').write_text(json.dumps(footprints,indent=2),encoding='utf-8')
review=pd.DataFrame([{'osm_building_id':f['properties']['osm_building_id'],
    'near_hospital':f['properties']['near_hospital'],
    'building_tag':f['properties']['building_tag'],
    'review_status':'UNREVIEWED', 'is_hospital_building':'UNKNOWN',
    'notes':''} for f in footprints['features']],
    columns=['osm_building_id','near_hospital','building_tag','review_status','is_hospital_building','notes'])
review.to_csv(OUT/'halifax_building_review_queue.csv',index=False)
print('Candidate building polygons:',len(footprints['features']))
display(review.head(15))

### 4. Map hospital sites and candidate footprints
Blue dots = reported hospital reference sites; orange outlines = **unreviewed buildings**. For visual review, use this alongside a suitable, licensed high-resolution orthophoto source. OpenStreetMap tiles are only a contextual basemap.

In [ ]:
m=folium.Map(location=CENTER,zoom_start=12,tiles='OpenStreetMap')
for _,p in sites.iterrows():
    folium.CircleMarker([p.lat,p.lon],radius=5,color='blue',fill=True,fill_opacity=0.9,
        tooltip=f"OSM hospital site: {p['name']}").add_to(m)
if footprints['features']:
    folium.GeoJson(footprints, name='Unreviewed building candidates',
      style_function=lambda _: {'color':'#ff8500','weight':2,'fillOpacity':0.08},
      tooltip=folium.GeoJsonTooltip(fields=['osm_building_id','near_hospital','building_tag','review_status'])).add_to(m)
folium.LayerControl().add_to(m)
m.save(str(OUT/'halifax_review_map.html'))
print('Saved permanent files:', sorted(p.name for p in OUT.iterdir()))
m

### 5. Why imagery is the gating item for model training
- **Sentinel-2 RGB = 10 m pixels**: adequate for a regional preview, generally not for building-by-building hospital detection.
- **GeoNOVA** reports roughly 20 cm provincial orthophotos, but downloads and licensing/access can be restricted. Confirm coverage, permissions, and cost before using images for training. Official information: [GeoNOVA base mapping](https://geonova.novascotia.ca/base-mapping), [orthophoto data access](https://geonova.novascotia.ca/new-orthophotography-now-available).
- Open Canadian aerial imagery includes **historic 1947/1977 Halifax mosaics** (NRCan), but those cannot be assumed to represent present-day hospital footprints: [Government of Canada Halifax collection](https://open.canada.ca/data/en/dataset/4e8e3c6a-c961-4def-bdc7-f24823462818).

**Definition of success:** a non-empty site CSV, candidate footprints GeoJSON and CSV (if Overpass succeeds), and review map. **No trained AI model or validated building boxes yet.**

**Next milestone:** source licensed modern imagery at <=1 m/pixel, verify building labels manually, cut training patches and convert checked geometry to bounding boxes, then train/evaluate an object detector. Attribution: © OpenStreetMap contributors (ODbL).
